# Job Application Assistant
A pipeline that analyses a job spec against your CV, identifies gaps, elicits missing evidence through targeted questions, and drafts a personal statement.

**Built on:** `llm.py` — swap `PROVIDER` to switch between Ollama (local/private), Groq (free cloud), or Anthropic.

## Pipeline
1. [Setup](#1-setup)
2. [Ingest — job spec + CV](#2-ingest)
3. [Parse — structured extraction](#3-parse)
4. [Analyse — gap analysis](#4-analyse)
5. [Elicit — targeted questions](#5-elicit)
6. [Draft — personal statement](#6-draft)
7. [Iterate — refine](#7-iterate)
8. [Export — save session](#8-export)

---
> **Privacy note:** With `PROVIDER = "ollama"` nothing leaves your machine. Switch to a cloud provider only if you're comfortable with that tradeoff.


---
# 1. Setup

In [1]:
# If running on Kaggle, install dependencies and set up Ollama
# If running locally with Ollama already running, skip to the imports

import os, sys

ON_KAGGLE = os.path.exists("/kaggle")

if ON_KAGGLE:
    import subprocess, time, requests

    # Install Ollama
    if not os.path.exists("/usr/local/bin/ollama"):
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, check=True, capture_output=True)

    # Start server
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Wait until ready
    print("Starting Ollama", end="")
    for _ in range(30):
        try:
            if requests.get("http://localhost:11434").status_code == 200:
                print(" ✅ Ready")
                break
        except:
            print(".", end="", flush=True)
            time.sleep(1)

    # Pull model
    subprocess.run(["ollama", "pull", "qwen2.5:7b"], check=True)
    print("✅ Model ready")
else:
    print("Running locally — make sure Ollama is running: ollama serve")


Running locally — make sure Ollama is running: ollama serve


In [2]:
# ── Import llm.py ─────────────────────────────────────────────────────────────
# On Kaggle: upload llm.py as a dataset or paste it inline
# Locally: ensure llm.py is in the same directory

try:
    from llm import chat, Conversation, extract_json, parse_json, use_provider, DEFAULT_MODEL
    print(f"✅ llm.py loaded — provider ready")
except ImportError:
    print("❌ llm.py not found. Upload it to /kaggle/input/ or the same directory.")
    raise


✅ llm.py loaded — provider ready


In [3]:
# ── Provider config ────────────────────────────────────────────────────────────
# Change this one variable to switch provider. Everything else stays the same.

PROVIDER = "ollama"   # "ollama" | "groq" | "anthropic" | "openai"

use_provider(PROVIDER)
print(f"Provider: {PROVIDER} | Model: {DEFAULT_MODEL}")


✅ Switched to ollama — default model: qwen2.5:7b
Provider: ollama | Model: qwen2.5:7b


---
# 2. Ingest — Job Spec + CV

Paste your job spec and CV as text. PDF/docx parsing added in the next section.

In [4]:
# ── Paste your job spec here ──────────────────────────────────────────────────
JOB_SPEC = """
PASTE JOB SPEC TEXT HERE

Include: job title, responsibilities, person spec / required experience,
behaviours or competencies being assessed, word limits if stated.
"""

# ── Paste your CV here ────────────────────────────────────────────────────────
CV = """
PASTE CV TEXT HERE

Include: employment history, skills, education, notable projects/achievements.
"""

print(f"Job spec: {len(JOB_SPEC.split())} words")
print(f"CV: {len(CV.split())} words")


Job spec: 23 words
CV: 11 words


In [5]:
# ── Optional: load from file ──────────────────────────────────────────────────
# Uncomment and adapt if you have files available

# from pathlib import Path
# JOB_SPEC = Path("job_spec.txt").read_text()
# CV = Path("cv.txt").read_text()

# ── Optional: parse PDF ───────────────────────────────────────────────────────
# !pip install pypdf python-docx -q

# import pypdf
# def read_pdf(path):
#     reader = pypdf.PdfReader(path)
#     return "\n".join(p.extract_text() for p in reader.pages if p.extract_text())

# import docx
# def read_docx(path):
#     doc = docx.Document(path)
#     return "\n".join(p.text for p in doc.paragraphs if p.text.strip())

# JOB_SPEC = read_pdf("job_spec.pdf")
# CV = read_docx("cv.docx")


---
# 3. Parse — Structured Extraction

Extract structured data from both documents so the analysis has clean inputs.

In [6]:
import json

def extract_json_robust(prompt: str, schema: str, retries: int = 2) -> dict:
    """Extract JSON from LLM response with retry on parse failure."""
    system = (
        "You return ONLY valid JSON matching the schema provided. "
        "No explanation, no markdown fences, no extra text. Just the JSON object."
    )
    for attempt in range(retries + 1):
        raw = chat(prompt, system=system, temperature=0.1)
        try:
            text = raw.strip().strip("```json").strip("```").strip()
            if not text.startswith("{"):
                text = text[text.find("{"):text.rfind("}")+1]
            return json.loads(text)
        except json.JSONDecodeError as e:
            if attempt < retries:
                print(f"Parse failed (attempt {attempt+1}), retrying... {e}")
            else:
                print(f"❌ Could not parse JSON after {retries+1} attempts")
                print("Raw response:", raw[:500])
                return {}


In [8]:
# ── Parse job spec ────────────────────────────────────────────────────────────

JOB_SPEC_SCHEMA = json.dumps({
    "job_title": "string",
    "organisation": "string",
    "sift_criteria": ["list of what is assessed at sift stage"],
    "required_experience": ["list of required experience items"],
    "technical_skills": ["list of specific technical skills or tools named"],
    "behaviours": ["list of behaviours or competencies assessed"],
    "word_limit": "integer or null",
    "role_summary": "2-3 sentence summary of what the role involves"
})

parsed_job = extract_json_robust(
    f"Extract structured information from this job specification:\n\n{JOB_SPEC}",
    JOB_SPEC_SCHEMA
)

print("✅ Job spec parsed:")
print(json.dumps(parsed_job, indent=2))


APIConnectionError: Connection error.

In [ ]:
# ── Parse CV ──────────────────────────────────────────────────────────────────

CV_SCHEMA = json.dumps({
    "name": "string",
    "current_role": "string",
    "current_employer": "string",
    "employment_history": [{"role": "", "employer": "", "duration": "", "key_achievements": []}],
    "technical_skills": ["list of tools, languages, frameworks"],
    "domains": ["list of subject/domain areas worked in"],
    "qualifications": ["list of degrees, certs, courses"],
    "publications_or_research": "summary or null"
})

parsed_cv = extract_json_robust(
    f"Extract structured information from this CV:\n\n{CV}",
    CV_SCHEMA
)

print("✅ CV parsed:")
print(json.dumps(parsed_cv, indent=2))


---
# 4. Analyse — Gap Analysis

Compare the job requirements against the CV. Identify what's strong, what's weak, and what's missing.

In [ ]:
# ── Gap analysis with chain of thought ───────────────────────────────────────

GAP_SCHEMA = json.dumps({
    "strong_matches": [{"requirement": "", "evidence": "", "strength": "high|medium"}],
    "partial_matches": [{"requirement": "", "evidence": "", "gap": "what is missing or thin"}],
    "missing": [{"requirement": "", "why_it_matters": ""}],
    "overall_fit_score": "integer 0-100",
    "fit_summary": "2-3 sentence honest assessment",
    "priority_gaps": ["top 3 gaps to address in the personal statement"]
})

gap_prompt = f"""You are an expert recruitment consultant assessing a job application.

JOB REQUIREMENTS:
{json.dumps(parsed_job, indent=2)}

CANDIDATE CV:
{json.dumps(parsed_cv, indent=2)}

Perform a thorough gap analysis. Be honest — identify genuine strengths but also 
real gaps. Consider both explicit requirements and implicit expectations for the role level.
Return your analysis as JSON."""

print("Running gap analysis...")
gap_analysis = extract_json_robust(gap_prompt, GAP_SCHEMA)

print("\n✅ Gap analysis complete")
print(f"\nOverall fit: {gap_analysis.get('overall_fit_score', '?')}/100")
print(f"\nSummary: {gap_analysis.get('fit_summary', '')}")
print(f"\nPriority gaps to address:")
for g in gap_analysis.get('priority_gaps', []):
    print(f"  • {g}")


In [ ]:
# ── Display full gap analysis ─────────────────────────────────────────────────

print("=" * 60)
print("STRONG MATCHES")
print("=" * 60)
for m in gap_analysis.get('strong_matches', []):
    print(f"\n✅ {m.get('requirement', '')}")
    print(f"   Evidence: {m.get('evidence', '')}")

print("\n" + "=" * 60)
print("PARTIAL MATCHES — thin evidence")
print("=" * 60)
for m in gap_analysis.get('partial_matches', []):
    print(f"\n~ {m.get('requirement', '')}")
    print(f"  Evidence: {m.get('evidence', '')}")
    print(f"  Gap: {m.get('gap', '')}")

print("\n" + "=" * 60)
print("MISSING — not evidenced")
print("=" * 60)
for m in gap_analysis.get('missing', []):
    print(f"\n❌ {m.get('requirement', '')}")
    print(f"   Why it matters: {m.get('why_it_matters', '')}")


---
# 5. Elicit — Targeted Questions

The elicitor uses the gap analysis to generate targeted questions, then conducts a structured conversation to draw out evidence the CV doesn't capture.

This is the core differentiator from simple cover letter generators — it elicits the right detail rather than just summarising what's already written.


In [ ]:
# ── Generate targeted questions based on gaps ─────────────────────────────────

QUESTIONS_SCHEMA = json.dumps([
    {
        "gap": "the requirement being addressed",
        "question": "the specific question to ask",
        "why": "what this question is trying to surface",
        "follow_up": "a potential follow-up if the answer is vague"
    }
])

questions_prompt = f"""Based on this gap analysis, generate targeted elicitation questions 
for the candidate. Focus on the priority gaps and partial matches.

Questions should:
- Surface specific examples and evidence not visible in the CV
- Be concrete enough to get useful answers
- Avoid yes/no questions
- Be ordered from most to least important

GAP ANALYSIS:
{json.dumps(gap_analysis, indent=2)}

Generate 5-7 questions as a JSON array."""

system_q = (
    "You return ONLY a valid JSON array. "
    "No explanation, no markdown, no extra text."
)

raw_q = chat(questions_prompt, system=system_q, temperature=0.2)
try:
    text = raw_q.strip().strip("```json").strip("```").strip()
    if not text.startswith("["):
        text = text[text.find("["):text.rfind("]")+1]
    elicitation_questions = json.loads(text)
    print(f"✅ Generated {len(elicitation_questions)} questions\n")
    for i, q in enumerate(elicitation_questions, 1):
        print(f"Q{i}: {q.get('question', '')}")
        print(f"     → Targeting: {q.get('gap', '')}\n")
except Exception as e:
    print(f"❌ Failed to parse questions: {e}")
    print(raw_q[:500])
    elicitation_questions = []


In [ ]:
# ── Run elicitation conversation ──────────────────────────────────────────────
# The elicitor asks each question and stores answers
# Answers feed directly into the PS drafting step

elicitor = Conversation(system="""You are an expert job application coach helping a candidate 
prepare a strong personal statement. Your role is to ask targeted questions to surface 
evidence and examples that support their application. 

Be encouraging but precise — ask for specifics when answers are vague.
If an answer is strong, acknowledge it briefly before moving on.
If an answer is vague, probe with the follow-up question.""")

collected_evidence = []

print("=" * 60)
print("ELICITATION SESSION")
print("Starting conversation — answer each question in the input box")
print("Type 'skip' to skip a question, 'done' to end early")
print("=" * 60)

for i, q_data in enumerate(elicitation_questions, 1):
    question = q_data.get('question', '')
    follow_up = q_data.get('follow_up', '')
    gap = q_data.get('gap', '')
    
    print(f"\n[Question {i}/{len(elicitation_questions)}]")
    print(f"Gap being addressed: {gap}")
    print(f"\n{question}")
    
    answer = input("Your answer: ").strip()
    
    if answer.lower() == 'done':
        print("Ending elicitation early.")
        break
    
    if answer.lower() == 'skip':
        print("Skipped.")
        continue
    
    # Check if answer needs probing
    if len(answer.split()) < 20:
        print(f"\nFollow-up: {follow_up}")
        follow_up_answer = input("Your answer: ").strip()
        if follow_up_answer.lower() not in ('skip', 'done'):
            answer = f"{answer}. {follow_up_answer}"
    
    collected_evidence.append({
        "gap": gap,
        "question": question,
        "answer": answer
    })
    
    # Brief acknowledgement from coach
    ack = elicitor.say(
        f"The candidate answered this question: '{question}'\n\nAnswer: {answer}\n\n"
        f"Give a ONE sentence acknowledgement only, then say 'Moving on.' — do not ask anything else."
    )
    print(f"\nCoach: {ack}")

print(f"\n✅ Elicitation complete — {len(collected_evidence)} answers collected")


---
# 6. Draft — Personal Statement

Use the gap analysis and elicited evidence to draft a personal statement. Targets the word limit extracted from the job spec.

In [ ]:
# ── Draft personal statement ──────────────────────────────────────────────────

word_limit = parsed_job.get('word_limit') or 500
sift_criteria = parsed_job.get('sift_criteria', [])

evidence_text = "\n\n".join([
    f"Re: {e['gap']}\nQ: {e['question']}\nA: {e['answer']}"
    for e in collected_evidence
])

draft_prompt = f"""You are an expert job application writer. Draft a personal statement 
for this candidate using ONLY the evidence provided. Do not invent details.

ROLE BEING APPLIED FOR:
{parsed_job.get('job_title', '')} at {parsed_job.get('organisation', '')}

WHAT THE SIFT ASSESSES:
{json.dumps(sift_criteria, indent=2)}

CANDIDATE STRENGTHS (from gap analysis):
{json.dumps([m['requirement'] + ': ' + m['evidence'] 
             for m in gap_analysis.get('strong_matches', [])], indent=2)}

ELICITED EVIDENCE:
{evidence_text}

CV SUMMARY:
Current role: {parsed_cv.get('current_role', '')} at {parsed_cv.get('current_employer', '')}
Skills: {', '.join(parsed_cv.get('technical_skills', [])[:10])}

REQUIREMENTS:
- Word limit: {word_limit} words MAXIMUM
- Structure around the sift criteria above
- Use specific examples throughout — no vague claims
- Write in first person, confident and direct tone
- Do not use bullet points — flowing prose only
- Do not overclaim experience that is thin in the evidence

Return the personal statement as plain text only."""

print("Drafting personal statement...")
draft = chat(draft_prompt, temperature=0.4)

word_count = len(draft.split())
print(f"\n✅ Draft complete — {word_count} words (limit: {word_limit})")
print("\n" + "=" * 60)
print(draft)
print("=" * 60)


---
# 7. Iterate — Refine

Refine the draft through conversation. Give feedback and the writer will adjust.

In [ ]:
# ── Iterative refinement loop ─────────────────────────────────────────────────

writer = Conversation(system=f"""You are an expert personal statement editor. 
You have written a draft personal statement for a job application.
The word limit is {word_limit} words. The current draft has {len(draft.split())} words.

CURRENT DRAFT:
{draft}

When the user gives feedback:
- Make only the changes requested
- Maintain word count within {word_limit} words
- Keep the same confident, direct tone
- After each revision, state the new word count
- Return the FULL revised statement, not just the changed section""")

current_draft = draft
current_wc = len(draft.split())

print("Refinement mode. Give feedback to adjust the draft.")
print("Commands: 'done' to finish | 'count' to check word count | 'show' to reprint draft")
print("-" * 60)

while True:
    feedback = input("\nYour feedback: ").strip()
    
    if feedback.lower() == 'done':
        print("\n✅ Refinement complete.")
        break
    
    if feedback.lower() == 'count':
        print(f"Current word count: {current_wc}/{word_limit}")
        continue
    
    if feedback.lower() == 'show':
        print("\n" + "=" * 60)
        print(current_draft)
        print("=" * 60)
        continue
    
    if not feedback:
        continue
    
    revised = writer.say(feedback)
    current_wc = len(revised.split())
    
    print(f"\n[Word count: {current_wc}/{word_limit}]")
    print("\n" + "=" * 60)
    print(revised)
    print("=" * 60)
    
    # Only update if it looks like a full revised draft
    if len(revised.split()) > 100:
        current_draft = revised
    
    if current_wc > word_limit:
        print(f"⚠️  Over limit by {current_wc - word_limit} words — ask for cuts if needed")


---
# 8. Export — Save Session

Save the full session to JSON for persistence across runs, and export the final draft as a text file.

In [ ]:
import json
from datetime import datetime
from pathlib import Path

# ── Build session object ──────────────────────────────────────────────────────

session = {
    "metadata": {
        "created": datetime.now().isoformat(),
        "provider": PROVIDER,
        "model": DEFAULT_MODEL,
        "job_title": parsed_job.get('job_title', ''),
        "organisation": parsed_job.get('organisation', ''),
    },
    "parsed_job": parsed_job,
    "parsed_cv": parsed_cv,
    "gap_analysis": gap_analysis,
    "elicitation_questions": elicitation_questions,
    "collected_evidence": collected_evidence,
    "final_draft": current_draft,
    "word_count": len(current_draft.split()),
    "word_limit": word_limit,
}

# ── Save session JSON ─────────────────────────────────────────────────────────

output_dir = Path("/kaggle/working") if ON_KAGGLE else Path(".")
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
org = parsed_job.get('organisation', 'application').replace(' ', '_').lower()[:20]

session_path = output_dir / f"session_{org}_{timestamp}.json"
with open(session_path, "w") as f:
    json.dump(session, f, indent=2)
print(f"✅ Session saved: {session_path}")

# ── Save final draft as text ──────────────────────────────────────────────────

draft_path = output_dir / f"personal_statement_{org}_{timestamp}.txt"
with open(draft_path, "w") as f:
    f.write(f"Personal Statement — {parsed_job.get('job_title', '')}\n")
    f.write(f"{parsed_job.get('organisation', '')}\n")
    f.write(f"Word count: {len(current_draft.split())}/{word_limit}\n")
    f.write("=" * 60 + "\n\n")
    f.write(current_draft)
print(f"✅ Draft saved: {draft_path}")


In [ ]:
# ── Reload a previous session ─────────────────────────────────────────────────
# Run this cell to continue from a saved session

# SESSION_FILE = "session_dvsa_20260401_1430.json"

# with open(SESSION_FILE) as f:
#     session = json.load(f)

# parsed_job     = session['parsed_job']
# parsed_cv      = session['parsed_cv']
# gap_analysis   = session['gap_analysis']
# collected_evidence = session['collected_evidence']
# current_draft  = session['final_draft']
# word_limit     = session['word_limit']

# print(f"Session loaded: {session['metadata']['job_title']} at {session['metadata']['organisation']}")
# print(f"Draft: {session['word_count']} words")


---
## What's next

| Step | What to build |
|---|---|
| `parser.py` | Proper PDF/docx ingestion with pypdf + python-docx |
| `analyser.py` | Standalone gap analysis module with scoring |
| `elicitor.py` | State machine elicitor — smarter follow-up logic |
| `writer.py` | Word count enforcement, multi-format output |
| Streamlit app | Interactive UI wrapping this pipeline |
| RAG | Index past applications to suggest evidence from history |
